# Daily Temperatures

# Problem Statement

Given an array `temperatures` where:

```text
temperatures[i]
```

represents the temperature on day `i`, return an array where each position tells us how many days we must wait until a warmer temperature.

If there is no future day with a warmer temperature, return:

```text
0
```

### Input

An integer array:

```text
temperatures
```

### Output

An integer array containing the number of days until a warmer temperature.

### Examples

```text
Input:
[73,74,75,71,69,72,76,73]

Output:
[1,1,4,2,1,1,0,0]
```

Explanation:

- `73` → warmer day is `74` → `1` day
- `74` → warmer day is `75` → `1` day
- `75` → warmer day is `76` → `4` days
- `71` → warmer day is `72` → `2` days
- `69` → warmer day is `72` → `1` day
- `72` → warmer day is `76` → `1` day
- `76` → no warmer future day → `0`
- `73` → no warmer future day → `0`

# Problem Explanation

For every temperature, we need to find:

```text
The first temperature to the right that is greater.
```

For example:

```text
[73, 74, 75, 71, 69, 72, 76, 73]
```

For:

```text
71
```

we need the first greater temperature:

```text
72
```

which is two positions away.

So the problem is essentially:

```text
Next Greater Element
```

but instead of returning the greater value, we return:

```text
distance between the two positions
```

The challenge is doing this efficiently.

# Brute Force

For every day, scan all future days until we find a warmer temperature.

For example:

```text
71
```

we check:

```text
69
72
```

and stop at `72`.

For every element, we may scan almost the entire remaining array.

### Complexity

For `N` temperatures:

```text
Time → O(N²)
Space → O(1)
```

This becomes inefficient for large inputs.

We need to avoid repeatedly scanning the same future temperatures.

# Key Insight

Instead of asking:

```text
For this temperature, where is the next warmer day?
```

we can process the temperatures while maintaining the days that are still **waiting for a warmer temperature**.

Suppose:

```text
[73, 74]
```

When we see:

```text
74
```

we know that `74` is warmer than `73`.

Therefore:

```text
73 → solved
```

The Stack can store the indices of temperatures that have not yet found a warmer day.

When a warmer temperature arrives, it resolves those previous temperatures.

This is a Monotonic Stack pattern.

# Why Store Indices?

We need to return:

```text
number of days to wait
```

For example:

```text
index 3 → temperature 71
index 5 → temperature 72
```

The answer is:

```text
5 - 3 = 2
```

Therefore, we need the **indices**, not just the temperatures.

The Stack stores:

```text
indices
```

and we use:

```python
i - stack[-1]
```

to calculate the waiting time.

# Monotonic Stack

The Stack stores indices whose temperatures are in **decreasing order**.

For example:

```text
temperatures:

73, 75, 71, 69
```

The Stack may contain:

```text
75
71
69
```

When a warmer temperature appears:

```text
72
```

we resolve:

```text
69 → 72
71 → 72
```

The Stack therefore contains unresolved temperatures.

The rule is:

```text
While current temperature > temperature at stack top:

    The current day is the answer for stack top.
```

# Optimal Approach

Create:

```python
result = [0] * len(temperatures)
stack = []
```

The Stack stores indices.

For every index `i`:

```text
current = temperatures[i]
```

While:

```text
stack is not empty
AND
current > temperatures[stack[-1]]
```

the current day is warmer than the temperature represented by the Stack top.

Therefore:

```text
previous_index = stack.pop()
```

and:

```text
result[previous_index] = i - previous_index
```

Then push the current index:

```python
stack.append(i)
```

At the end, any indices still in the Stack have no warmer future day.

Their result remains:

```text
0
```

In [1]:
class Solution:

    def dailyTemperatures(self, temperatures: list[int]) -> list[int]:

        result = [0] * len(temperatures)

        stack = []

        for i in range(len(temperatures)):

            while (
                stack
                and temperatures[i] > temperatures[stack[-1]]
            ):
                previous_index = stack.pop()

                result[previous_index] = i - previous_index

            stack.append(i)

        return result

# Dry Run

Input:

```text
[73,74,75,71,69,72,76,73]
```

Start:

```text
result = [0,0,0,0,0,0,0,0]
stack = []
```

### Index 0

Temperature:

```text
73
```

Nothing to resolve.

Push index:

```text
stack = [0]
```

---

### Index 1

Temperature:

```text
74
```

Compare:

```text
74 > 73
```

Yes.

Pop index `0`.

Distance:

```text
1 - 0 = 1
```

Therefore:

```text
result[0] = 1
```

Push `1`:

```text
stack = [1]
```

---

### Index 2

Temperature:

```text
75
```

Compare:

```text
75 > 74
```

Yes.

Pop index `1`.

```text
result[1] = 2 - 1 = 1
```

Push `2`.

```text
stack = [2]
```

---

### Index 3

Temperature:

```text
71
```

`71` is not greater than `75`.

Push:

```text
stack = [2,3]
```

---

### Index 4

Temperature:

```text
69
```

`69` is not greater than `71`.

Push:

```text
stack = [2,3,4]
```

---

### Index 5

Temperature:

```text
72
```

Compare with `69`:

```text
72 > 69
```

Pop index `4`.

```text
result[4] = 5 - 4 = 1
```

Compare with `71`:

```text
72 > 71
```

Pop index `3`.

```text
result[3] = 5 - 3 = 2
```

Compare with `75`:

```text
72 > 75
```

False.

Push `5`:

```text
stack = [2,5]
```

---

### Index 6

Temperature:

```text
76
```

Compare with `72`:

```text
76 > 72
```

Pop `5`.

```text
result[5] = 6 - 5 = 1
```

Compare with `75`:

```text
76 > 75
```

Pop `2`.

```text
result[2] = 6 - 2 = 4
```

Push `6`.

```text
stack = [6]
```

---

### Index 7

Temperature:

```text
73
```

`73` is not greater than `76`.

Push:

```text
stack = [6,7]
```

Final result:

```text
[1,1,4,2,1,1,0,0]
```

# Why Do We Keep Decreasing Temperatures?

Suppose:

```text
[75, 71, 69]
```

The Stack contains:

```text
[75, 71, 69]
```

in decreasing temperature order.

Now suppose:

```text
72
```

arrives.

It is warmer than:

```text
69
```

so `69` is resolved.

It is also warmer than:

```text
71
```

so `71` is resolved.

But it is not warmer than:

```text
75
```

so `75` remains unresolved.

This is exactly why the Stack works.

A new temperature can resolve one or multiple previous temperatures.

# Why Do We Store Unresolved Indices?

Consider:

```text
[73,74,75]
```

After processing:

```text
73
```

we do not yet know its answer.

After seeing:

```text
74
```

we know:

```text
73 → 74
```

So `73` can be removed from the Stack.

Similarly:

```text
75
```

has no warmer temperature yet, so it remains.

Therefore the Stack represents:

```text
Temperatures waiting for a warmer day.
```

This is a useful way to think about many Monotonic Stack problems.

# Edge Cases

### Empty Array

```text
Input:
[]

Output:
[]
```

---

### One Temperature

```text
Input:
[70]

Output:
[0]
```

There is no future day.

---

### Strictly Increasing

```text
Input:
[70,71,72,73]
```

Output:

```text
[1,1,1,0]
```

Every day except the last has a warmer next day.

---

### Strictly Decreasing

```text
Input:
[73,72,71,70]
```

Output:

```text
[0,0,0,0]
```

No warmer future day exists.

---

### Equal Temperatures

```text
Input:
[70,70,71]
```

Output:

```text
[2,1,0]
```

Equal temperatures are not warmer.

The condition must therefore be:

```python
temperatures[i] > temperatures[stack[-1]]
```

not:

```python
>=
```

# Common Mistakes

### Mistake 1 — Storing Temperatures Instead of Indices

We need:

```text
number of days
```

so we need the positions.

Store:

```python
stack.append(i)
```

not:

```python
stack.append(temperatures[i])
```

---

### Mistake 2 — Using `>=`

The problem asks for a **warmer** temperature.

Therefore:

```text
72 > 72
```

is false.

Use:

```python
temperatures[i] > temperatures[stack[-1]]
```

---

### Mistake 3 — Scanning Forward Again

Once we use a Monotonic Stack, we should not scan forward for every element.

That would return us to:

```text
O(N²)
```

---

### Mistake 4 — Removing Elements Without Calculating Their Distance

When an index is popped:

```python
previous_index = stack.pop()
```

the answer is:

```python
i - previous_index
```

The current index is the first warmer day for that popped index.

# Complexity

Let:

```text
N = len(temperatures)
```

Every index is:

```text
Pushed once
Popped at most once
```

Therefore:

```text
Time → O(N)
```

The Stack can contain at most `N` indices:

```text
Space → O(N)
```

The result array also requires:

```text
O(N)
```

Therefore the total auxiliary/output space is:

```text
O(N)
```

Final complexity:

```text
Time  → O(N)
Space → O(N)
```

# Why Is It O(N)?

The nested `while` loop may look like:

```text
O(N²)
```

but it is not.

Each index enters the Stack once:

```text
push → once
```

and leaves the Stack at most once:

```text
pop → once
```

Therefore the total number of Stack operations is at most:

```text
2N
```

which is:

```text
O(N)
```

This is amortized analysis.

# Comparison

| Approach | Time | Space |
|---|---:|---:|
| Brute Force | O(N²) | O(1) |
| Monotonic Stack | O(N) | O(N) |

The Stack avoids repeatedly searching through the future.

Each temperature is processed only a constant number of times.

# Pattern Recognition

This is the classic:

```text
Next Greater Element
```

pattern.

When you see:

```text
For every element,
find the first greater element to the right.
```

immediately think:

```text
Monotonic Stack
```

For Daily Temperatures:

```text
Need next greater temperature
          ↓
Store unresolved indices
          ↓
Maintain decreasing temperatures
          ↓
Warmer temperature arrives
          ↓
Pop resolved indices
```

The only difference from a standard Next Greater Element problem is that we return:

```text
distance
```

instead of:

```text
greater value
```

# Connection With Previous Stack Problems

This problem connects several ideas we have already learned.

### Previous Problems

`Largest Rectangle in Histogram`

```text
Monotonic Stack
```

`Remove K Digits`

```text
Greedy + Monotonic Stack
```

`Asteroid Collision`

```text
Stack-based simulation
```

`Validate Stack Sequences`

```text
Direct Stack simulation
```

Now:

```text
Daily Temperatures
```

introduces one of the most important Monotonic Stack patterns:

```text
Next Greater Element
```

The important progression is:

```text
Stack
 ↓
Stack Simulation
 ↓
Monotonic Stack
 ↓
Next Greater / Smaller
```

# Takeaway

For every temperature, we need:

```text
The first warmer temperature to the right.
```

Instead of scanning forward for every day, maintain a Stack of:

```text
indices waiting for a warmer temperature
```

When a warmer temperature arrives:

```text
while current > stack top:

    previous = stack.pop()

    answer[previous] = current_index - previous
```

The key pattern is:

```text
Next Greater Element
        ↓
Monotonic Stack
```

Complexity:

```text
Time  → O(N)
Space → O(N)
```

The main lesson:

> If elements are waiting for the first larger/smaller element that appears later, a Monotonic Stack should be one of your first ideas.